# Kredi riskinde yöntemsel sağlamlık ve ikinci veri kümesi: Colab deney paketi

**Amaç:** Mevcut makaledeki UCI kredi kartı verisini ve *South German Credit* veri kümesini aynı deney protokolü altında karşılaştırmak; bağımsız dış katman değerlendirmesinde kalibrasyonun, hata maliyetinin ve sınıf ağırlıklarının etkisini incelemek.

**Önemli:** Bu çalışma önceki notebook'taki sonuçları **değiştirmeden** ayrı bir yeniden doğrulama çalışması olarak çalışır. Kod çalıştırılmadan burada yeni deney sonuçları iddia edilmemektedir. `MODE='FAST'` başlangıç testidir; makaleye girecek sonuçlar için `MODE='PAPER'` ile yeniden çalıştırın. Bu notebook'ta veri kümeleri arasında model transferi yapılmamakta, aynı **protokol** veri kümelerinde ayrı ayrı sınanmaktadır.

**Dikkat:** South German Credit (n=1.000), 1970'lere ait kredi riskini temsil eder. Bu, güncel kredi kartı temerrüdüne doğrudan dış geçerlik kanıtı değil; farklı bağlamda yöntemsel sağlamlık kontrolüdür. UCI kredi kartı veri kümesi önceki çalışma sırasında zaten incelendiğinden yeni sonuçlar tam anlamıyla ön-kayıtlı bağımsız doğrulama değildir.
**Güncelleme (veri erişimi):** South German Credit verisi `fetch_ucirepo(id=573)` yerine UCI'nin resmî ZIP arşivinden yüklenir. Veri ve hedef kodlama doğrulaması korunur.


In [ ]:
%pip -q install "ucimlrepo>=0.0.7" "xgboost>=2.0,<4" "scikit-learn>=1.4,<2" "openpyxl>=3.1"

## 1. Deney ayarları

`FAST`: 3 dış kat / 3 iç kat / 4 arama denemesi; yalnız üç temel model. `PAPER`: 5 dış kat / 3 iç kat / 8 arama denemesi; ağırlık ablasyonu dahil beş model. Sonuçlar ve ortam sürümleri CSV/JSON olarak kaydedilir. İç CV'de model seçimi dış test katına erişmez; eşik seçimi yalnızca dış eğitim grubunda OOF tahminleriyle yapılır. Sınıf ağırlığı seçenekleri modelin bir parçasıdır, bu nedenle ağırlık etkisi ayrı bir deney faktörüdür.

In [ ]:
import sys, json, os, platform, warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
import scipy, sklearn, xgboost
from ucimlrepo import fetch_ucirepo
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss, f1_score, precision_score, recall_score, accuracy_score, confusion_matrix
from IPython.display import display
warnings.filterwarnings("ignore", category=FutureWarning)

MODE = "FAST"  # Son makale analizi öncesi "PAPER" yapın.
assert MODE in ("FAST", "PAPER")
RANDOM_STATE = 42
OUTER_FOLDS = 3 if MODE == "FAST" else 5
INNER_FOLDS = 3
SEARCH_ITER = 4 if MODE == "FAST" else 8
BOOTSTRAPS = 500 if MODE == "FAST" else 3000
FN_COST_RATIOS = [1, 2, 3, 5, 10]
OUTPUT_DIR = Path('/content/kredi_revizyon_q1q3') if Path('/content').exists() else Path('/mnt/data/kredi_revizyon_q1q3')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'MODE={MODE}; outer={OUTER_FOLDS}; inner={INNER_FOLDS}; output={OUTPUT_DIR}')

## 2. Veriyi yükleme, hedef değişkeni kontrol etme

**UCI içe aktarma düzeltmesi:** `South German Credit` (UCI id=573) `ucimlrepo` ile mevcut olmasına rağmen doğrudan içe aktarılamayabilir. Bu nedenle resmî UCI ZIP dosyasındaki `SouthGermanCredit.asc` dosyası okunur. Kod hem Almanca hem İngilizce sütun adlarını normalize eder ve sınıf oranını kontrol eder.

South German Credit veri kümesinde `credit_risk=0` (orijinal `kredit=0`) **kötü kredi**, `credit_risk=1` **iyi kredi** anlamına gelir. Pozitif sınıfı **riskli müşteri (1)** olarak standardize ediyoruz. `duration`, `amount` ve `age` sayısal değişken, kalan 17 alan kategorik/ordinal kod olarak işlenir.

Kaynak: https://archive.ics.uci.edu/dataset/573/south+german+credit (UCI; CC BY 4.0; DOI 10.24432/C5QG88).


In [ ]:
from io import BytesIO
from zipfile import ZipFile, BadZipFile
import requests

# Resmî UCI veri dosyası: id=573 `ucimlrepo` ile içe aktarılamadığında kullanılır.
SOUTH_GERMAN_URL = 'https://archive.ics.uci.edu/static/public/573/south+german+credit+update.zip'
SG_DE_COLUMNS = [
    'laufkont','laufzeit','moral','verw','hoehe','sparkont','beszeit','rate',
    'famges','buerge','wohnzeit','verm','alter','weitkred','wohn','bishkred',
    'beruf','pers','telef','gastarb','kredit'
]
SG_EN_COLUMNS = [
    'status','duration','credit_history','purpose','amount','savings',
    'employment_duration','installment_rate','personal_status_sex',
    'other_debtors','present_residence','property','age',
    'other_installment_plans','housing','number_credits','job',
    'people_liable','telephone','foreign_worker','credit_risk'
]
SG_RENAME = dict(zip(SG_DE_COLUMNS, SG_EN_COLUMNS))


def load_south_german_credit():
    try:
        response = requests.get(SOUTH_GERMAN_URL, timeout=60)
        response.raise_for_status()
        with ZipFile(BytesIO(response.content)) as archive:
            files = [f for f in archive.namelist()
                     if Path(f).name.lower() == 'southgermancredit.asc']
            if len(files) != 1:
                raise ValueError('UCI arşivinde SouthGermanCredit.asc bulunamadı.')
            with archive.open(files[0]) as datafile:
                df = pd.read_csv(datafile, sep=r'\s+', engine='python')
    except (requests.RequestException, BadZipFile, OSError) as exc:
        raise RuntimeError('South German Credit ZIP dosyası UCI üzerinden okunamadı. '
                           'UCI sunucusunun erişimini kontrol edin: ' + SOUTH_GERMAN_URL) from exc

    # Veri dosyasında İngilizce/Germanca başlık veya başlıksız tablo olabilir.
    if df.shape == (999, 21) and pd.to_numeric(pd.Index(df.columns), errors='coerce').notna().all():
        # Başlıksız veride ilk gözlem başlık sanılmıştır: bütün veriyi tekrar oku.
        with ZipFile(BytesIO(response.content)) as archive:
            with archive.open(files[0]) as datafile:
                df = pd.read_csv(datafile, sep=r'\s+', engine='python',
                                 header=None, names=SG_DE_COLUMNS)
    df.columns = [str(c).strip() for c in df.columns]
    if list(df.columns) == SG_DE_COLUMNS:
        df = df.rename(columns=SG_RENAME)
    if set(df.columns) != set(SG_EN_COLUMNS):
        raise ValueError('South German Credit sütun adları beklenen şemayla uyuşmuyor: '
                         + str(df.columns.tolist()))
    df = df[SG_EN_COLUMNS].apply(pd.to_numeric, errors='raise').reset_index(drop=True)
    if df.shape != (1000, 21) or df.isna().any().any():
        raise ValueError(f'German veri boyutu veya eksik değerler beklenenden farklı: {df.shape}')
    y_raw = df.pop('credit_risk').astype(int)
    if set(y_raw.unique()) != {0,1} or y_raw.sum() != 700:
        raise ValueError('German hedef kodları değişmiş olabilir; 700 iyi/300 kötü bekleniyordu.')
    y = (y_raw == 0).astype(int)  # 0 kötü kredi -> pozitif risk sınıfı = 1
    X = df
    categorical = [col for col in X.columns if col not in ('duration','amount','age')]
    assert X.shape == (1000, 20) and int(y.sum()) == 300
    return X, y, categorical


def load_datasets():
    tai = fetch_ucirepo(id=350)
    Xt = tai.data.features.copy()
    yt = tai.data.targets.iloc[:, 0].astype(int).reset_index(drop=True)
    Xt = Xt.drop(columns=['ID'], errors='ignore').reset_index(drop=True)
    Xt.columns = ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE',
                  'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6',
                  'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
                  'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']
    assert Xt.shape == (30000, 23)
    assert set(yt.unique()) == {0, 1}
    assert 0.20 < yt.mean() < 0.24, 'Taiwan hedef oranı beklenenden farklı.'

    Xg, yg, g_categories = load_south_german_credit()
    data = {
        'Taiwan_Credit_Default': (Xt, yt, ['SEX', 'EDUCATION', 'MARRIAGE']),
        'South_German_Credit': (Xg, yg, g_categories)
    }
    for key, (X, y, categoricals) in data.items():
        if len(X) != len(y):
            raise ValueError('Girdi/etiket boyu uyuşmuyor.')
        if X.isna().any().any():
            print(key, 'eksik değer:', int(X.isna().sum().sum()))
        print(key, 'X=', X.shape, 'risk_n=', int(y.sum()),
              'risk_rate=', round(y.mean(),4),
              'kategorik_sutun=',len(categoricals))
    return data

datasets = load_datasets()


## 3. Ön işleme, modeller ve metrikler

Kategorik değerler eğitim verisiyle öğrenilen one-hot kodlamaya tabi tutulur. Ölçekleme ve imputasyon `Pipeline` içinde olduğundan dış katlarda veri sızıntısı oluşmaz. Ham ve sigmoid-kalibre olasılıklar **aynı dış test katında** karşılaştırılır. `ensemble=False`, kalibrasyonla farklı bir model topluluğu oluşturmanın etkisini sınırlar.

In [ ]:
def make_preprocessor(X, categoricals):
    numeric = [c for c in X.columns if c not in categoricals]
    numeric_pipeline = Pipeline([('imp', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
    categorical_pipeline = Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                                     ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
    return ColumnTransformer([('num', numeric_pipeline, numeric),
                              ('cat', categorical_pipeline, categoricals)], sparse_threshold=0.0)


def model_configs(X, cats, y_outer):
    prep = make_preprocessor(X, cats)
    pos_weight = float((y_outer == 0).sum() / max(1, (y_outer == 1).sum()))
    conf = {
      'LR': (Pipeline([('pre', clone(prep)), ('clf', LogisticRegression(max_iter=2500, solver='liblinear', random_state=42))]),
             {'clf__C':[0.1, 1.0, 10.0], 'clf__class_weight':[None, 'balanced']}),
      'RF_Unweighted': (Pipeline([('pre', clone(prep)), ('clf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=1))]),
             {'clf__n_estimators':[160,300], 'clf__max_depth':[8,16,None], 'clf__min_samples_leaf':[1,4]}),
      'XGB_Unweighted': (Pipeline([('pre', clone(prep)), ('clf', xgboost.XGBClassifier(objective='binary:logistic',
          eval_metric='logloss', tree_method='hist', n_jobs=1, random_state=42))]),
             {'clf__n_estimators':[160,300], 'clf__max_depth':[3,5], 'clf__learning_rate':[0.03,0.08],
              'clf__subsample':[0.8,1.0], 'clf__colsample_bytree':[0.8,1.0]})
    }
    if MODE == 'PAPER':
        conf['RF_Weighted'] = (Pipeline([('pre', clone(prep)), ('clf', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=1))]),
             {'clf__n_estimators':[160,300], 'clf__max_depth':[8,16,None], 'clf__min_samples_leaf':[1,4]})
        conf['XGB_Weighted'] = (Pipeline([('pre', clone(prep)), ('clf', xgboost.XGBClassifier(objective='binary:logistic',
              eval_metric='logloss', scale_pos_weight=pos_weight, tree_method='hist', n_jobs=1, random_state=42))]),
             {'clf__n_estimators':[160,300], 'clf__max_depth':[3,5], 'clf__learning_rate':[0.03,0.08],
              'clf__subsample':[0.8,1.0], 'clf__colsample_bytree':[0.8,1.0]})
    return conf


def choose_threshold(y, prob, cfn=5):
    y = np.asarray(y, dtype=int)
    prob = np.asarray(prob, dtype=float)
    # 0 ve 1 dahil: her zaman negatif/pozitif politika da adaydır
    grid = np.unique(np.r_[0.0, np.linspace(0.005, 0.995, 199), 1.0])
    costs = np.array([np.sum((y == 0) & (prob >= t)) + cfn*np.sum((y == 1) & (prob < t)) for t in grid])
    return float(grid[np.argmin(costs)])


def confusion_cost(y, p, threshold, cfn=5):
    pred = (np.asarray(p) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    return int(fp), int(fn), int(fp + cfn*fn)


def ece_score(y, p, n_bins=10):
    # nicelik tabanlı (equal-frequency) ECE, yalnızca yaklaşık güvenilirlik özeti
    frame = pd.DataFrame({'y':np.asarray(y), 'p':np.asarray(p)})
    frame['bin'] = pd.qcut(frame.p.rank(method='first'), q=min(n_bins, len(frame)), labels=False)
    agg = frame.groupby('bin').agg(n=('y','size'), actual=('y','mean'), predicted=('p','mean'))
    return float((agg.n/len(frame) * abs(agg.actual-agg.predicted)).sum())


def metric_row(y,p):
    return dict(AUC=roc_auc_score(y,p), AP=average_precision_score(y,p),
                Accuracy050=accuracy_score(y,np.asarray(p)>=.5),
                Precision050=precision_score(y,np.asarray(p)>=.5,zero_division=0),
                Recall050=recall_score(y,np.asarray(p)>=.5,zero_division=0),
                F1050=f1_score(y,np.asarray(p)>=.5,zero_division=0),
                Brier=brier_score_loss(y,p), LogLoss=log_loss(y,np.clip(p,1e-12,1-1e-12)),
                ECE10=ece_score(y,p))


def build_calibrator(base, cv):
    try:
        return CalibratedClassifierCV(estimator=clone(base), method='sigmoid', cv=cv, ensemble=False)
    except TypeError:
        return CalibratedClassifierCV(base_estimator=clone(base), method='sigmoid', cv=cv, ensemble=False)

# Matematiksel kontroller
assert choose_threshold(np.array([0,1]), np.array([0.1,0.9]), cfn=5) >= 0
assert confusion_cost([0,1], [0.9,0.1], .5, 5) == (1,1,6)
print('Yardımcı fonksiyon doğrulamaları geçti.')

## 4. Dış katlı değerlendirme — çalıştırılacak ana deney

Bu hücre zaman alır. FAST sürümünün amacı yalnızca kodun tamamını test etmek; makale tablolarını **PAPER** sürümünden oluşturmalısınız. Her katın sonunda CSV kayıtları alınır. Kalibre eşik, **yalnızca dış eğitim kümesinde tekrar katlamalı kalibre OOF tahminlerinden** seçilir. Dış testte hiçbir eşik/model/kalibrasyon seçilmez.

In [ ]:
fold_rows, decision_rows, pooled_rows, tuning_rows = [], [], [], []
start_time = time.time()
for dataset_name, (X, y, cats) in datasets.items():
    outer_cv = StratifiedKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    for fold_no, (itr, ite) in enumerate(outer_cv.split(X,y), start=1):
        Xt, Xe = X.iloc[itr], X.iloc[ite]
        yt, ye = y.iloc[itr], y.iloc[ite]
        inner_cv = StratifiedKFold(n_splits=INNER_FOLDS, shuffle=True, random_state=RANDOM_STATE + fold_no)
        conf = model_configs(Xt, cats, yt)
        for model_name, (base, dist) in conf.items():
            print(f'{dataset_name} | fold {fold_no}/{OUTER_FOLDS} | {model_name}', flush=True)
            search = RandomizedSearchCV(base, dist, n_iter=SEARCH_ITER, scoring='roc_auc',
                 cv=inner_cv, n_jobs=-1, random_state=RANDOM_STATE + fold_no, refit=True,
                 error_score='raise', verbose=0)
            search.fit(Xt, yt)
            best = search.best_estimator_
            calibrator = build_calibrator(best, inner_cv)
            # Tam dış eğitim verisinden çapraz doğrulamalı eşik tahmini
            train_p_raw = cross_val_predict(clone(best), Xt, yt, cv=inner_cv,
                                             method='predict_proba', n_jobs=-1)[:,1]
            train_p_cal = cross_val_predict(build_calibrator(best, inner_cv), Xt, yt,
                                             cv=inner_cv, method='predict_proba', n_jobs=-1)[:,1]
            # Dış test kümesindeki modelin fit edilmesi; calibrator ensemble=False
            fitted_raw = clone(best).fit(Xt, yt)
            calibrator.fit(Xt, yt)
            test_p_raw = fitted_raw.predict_proba(Xe)[:,1]
            test_p_cal = calibrator.predict_proba(Xe)[:,1]
            for probability_type, train_p, test_p in [('raw',train_p_raw,test_p_raw),
                                                       ('calibrated',train_p_cal,test_p_cal)]:
                results = metric_row(ye, test_p)
                fold_rows.append(dict(Dataset=dataset_name,Fold=fold_no,Model=model_name,
                                      Probability=probability_type,Test_N=len(ye),Default_Rate=float(ye.mean()),**results))
                for cost in FN_COST_RATIOS:
                    selected_t = choose_threshold(yt, train_p, cfn=cost)
                    fpr, fnr, c = confusion_cost(ye, test_p, selected_t, cfn=cost)
                    decision_rows.append(dict(Dataset=dataset_name,Fold=fold_no,Model=model_name,
                                   Probability=probability_type,FN_Cost=cost,Policy='OOF_Selected',
                                   Threshold=selected_t,FP=fpr,FN=fnr,Total_Cost=c,Average_Cost=c/len(ye)))
                    if probability_type=='calibrated':
                        # Bayes beklenen kayıp altında, iyi kalibre edilmiş p varsayımı
                        theoretical_t = 1.0/(1.0+cost)
                        fp, fn, cc = confusion_cost(ye, test_p, theoretical_t, cfn=cost)
                        decision_rows.append(dict(Dataset=dataset_name,Fold=fold_no,Model=model_name,
                            Probability=probability_type,FN_Cost=cost,Policy='Bayes_Threshold',
                            Threshold=theoretical_t,FP=fp,FN=fn,Total_Cost=cc,Average_Cost=cc/len(ye)))
                for local_i, label_i, pi in zip(ite, ye.to_numpy(), test_p):
                    pooled_rows.append(dict(Dataset=dataset_name,RowID=int(local_i),Fold=fold_no,
                                            Model=model_name,Probability=probability_type,
                                            Y=int(label_i),P=float(pi)))
            tuning_rows.append(dict(Dataset=dataset_name,Fold=fold_no,Model=model_name,
                                   Inner_Best_AUC=search.best_score_,Best_Params=json.dumps(search.best_params_,sort_keys=True)))
            for frame, fn in [(fold_rows,'fold_metrics.csv'),(decision_rows,'decisions.csv'),
                               (pooled_rows,'pooled_predictions.csv'),(tuning_rows,'tuning.csv')]:
                pd.DataFrame(frame).to_csv(OUTPUT_DIR/fn,index=False)
print('Ana deney tamamlandı. Geçen dakika:',round((time.time()-start_time)/60,1))
display(pd.DataFrame(fold_rows).groupby(['Dataset','Model','Probability'])[['AUC','Brier','AP','ECE10']].mean().round(4))

## 5. Pooled out-of-fold sonuçlar ve eşleştirilmiş belirsizlik aralıkları

Her örnek dış test olarak bir kez görülmüştür. Bootstrap, dış katlarda elde edilmiş **sabit** tahmin vektörlerini koşullu olarak yeniden örnekler; katlar arası model eğitimi belirsizliğini tam yansıtmaz. Bu nedenle küçük farkların p-değerlerini kesin üstünlük kanıtı olarak yorumlamayın. Burada kalibrasyonun aynı modelde Brier farkını ve aynı maliyet politikasında gözlenen hata maliyetini ne kadar değiştirdiği özetlenir.

In [ ]:
dfp = pd.read_csv(OUTPUT_DIR/'pooled_predictions.csv')
dfd = pd.read_csv(OUTPUT_DIR/'decisions.csv')
assert not dfp.duplicated(['Dataset','Model','Probability','RowID']).any()
summary = (dfp.groupby(['Dataset','Model','Probability'],sort=True)
           .apply(lambda x:pd.Series(metric_row(x.Y.to_numpy(), x.P.to_numpy())),include_groups=False)
           .reset_index())
summary.to_csv(OUTPUT_DIR/'pooled_metrics.csv',index=False)
decision_summary = (dfd.groupby(['Dataset','Model','Probability','FN_Cost','Policy'],as_index=False)
                    .agg(FP=('FP','sum'),FN=('FN','sum'),Cost=('Total_Cost','sum'),N=('Total_Cost','size'),
                         Average_Cost=('Average_Cost','mean'),Threshold_Mean=('Threshold','mean'),
                         Threshold_SD=('Threshold','std')))
decision_summary.to_csv(OUTPUT_DIR/'cost_summary.csv',index=False)
display(summary.round(4))
display(decision_summary.query('FN_Cost==5').round(4))

In [ ]:
def paired_brier_interval(y,a,b, seed=42, n_boot=1000):
    rng = np.random.default_rng(seed)
    loss = (a-y)**2-(b-y)**2
    obs = float(loss.mean())
    boot = np.empty(n_boot)
    for i in range(n_boot):
        boot[i] = float(loss[rng.integers(0,len(loss),len(loss))].mean())
    return obs,float(np.quantile(boot,.025)),float(np.quantile(boot,.975))

paired_rows=[]
for dataset_name in dfp.Dataset.unique():
    for model in dfp[dfp.Dataset==dataset_name].Model.unique():
        z = dfp.query('Dataset==@dataset_name and Model==@model').pivot(index='RowID',columns='Probability',values='P')
        y = dfp.query('Dataset==@dataset_name and Model==@model and Probability=="raw"').set_index('RowID').loc[z.index].Y.to_numpy()
        diff,l,u = paired_brier_interval(y,z.raw.to_numpy(),z.calibrated.to_numpy(),
                                           n_boot=BOOTSTRAPS)
        paired_rows.append(dict(Dataset=dataset_name,Model=model,Contrast='Raw_Brier_minus_Calibrated_Brier',
                                Difference=diff,CI_Lower=l,CI_Upper=u))
paired = pd.DataFrame(paired_rows)
paired.to_csv(OUTPUT_DIR/'paired_brier_bootstrap.csv',index=False)
display(paired.round(5))
print('Not: Bu güven aralıkları öğrenme algoritmasının eğitim belirsizliğini bütünüyle kapsamaz.')

## 5.1. Kalibrasyonun karar maliyetine etkisi: eşleştirilmiş bootstrap

Her dış test satırı için kat bazında **eğitimden seçilmiş eşikler** kullanılarak hata maliyeti hesaplanır. Ham−kalibre farkının %95 aralığı, aynı kişileri birlikte yeniden örnekleyen bir bootstrap ile üretilir. Bu güven aralığı sabit dış kat tahminlerine koşulludur; yeni veri/kurumlara genellenebilirlik veya yeniden eğitim belirsizliği için doğrudan güven aralığı değildir.

In [ ]:
def cost_loss_each(y,p,t,cfn):
    y=np.asarray(y,dtype=int);p=np.asarray(p,dtype=float);t=np.asarray(t,dtype=float)
    return ((y==0)&(p>=t)).astype(float)+cfn*((y==1)&(p<t)).astype(float)

bootstrap_cost_rows=[]
for (dataset,model,cfn), sub in dfd.query('Policy=="OOF_Selected"').groupby(['Dataset','Model','FN_Cost']):
    preds=dfp[(dfp.Dataset==dataset)&(dfp.Model==model)]
    raw=preds[preds.Probability=='raw'][['RowID','Fold','Y','P']].rename(columns={'P':'p_raw'})
    cal=preds[preds.Probability=='calibrated'][['RowID','P']].rename(columns={'P':'p_cal'})
    tab=raw.merge(cal,on='RowID',validate='one_to_one')
    tr=sub[sub.Probability=='raw'][['Fold','Threshold']].rename(columns={'Threshold':'thr_raw'})
    tc=sub[sub.Probability=='calibrated'][['Fold','Threshold']].rename(columns={'Threshold':'thr_cal'})
    tab=tab.merge(tr,on='Fold',validate='many_to_one').merge(tc,on='Fold',validate='many_to_one')
    lr=cost_loss_each(tab.Y,tab.p_raw,tab.thr_raw,cfn)
    lc=cost_loss_each(tab.Y,tab.p_cal,tab.thr_cal,cfn)
    per_person_difference=lr-lc
    rng=np.random.default_rng(RANDOM_STATE)
    diffs=[]
    for _ in range(BOOTSTRAPS):
        ind=rng.integers(0,len(tab),len(tab))
        diffs.append(float(per_person_difference[ind].mean()))
    lo,hi=np.quantile(diffs,[.025,.975])
    bootstrap_cost_rows.append(dict(Dataset=dataset,Model=model,FN_Cost=cfn,
        Avg_Cost_Raw=float(lr.mean()),Avg_Cost_Calibrated=float(lc.mean()),
        Avg_Cost_Difference_RawMinusCal=float(per_person_difference.mean()),
        CI_Lower=float(lo),CI_Upper=float(hi),N=len(tab)))
cost_boot=pd.DataFrame(bootstrap_cost_rows)
cost_boot.to_csv(OUTPUT_DIR/'paired_cost_bootstrap.csv',index=False)
display(cost_boot.query('FN_Cost==5').round(5))
assert len(cost_boot)>0
print('Kalibre maliyet bootstrap tamamlandı.')

## 6. Deney manifesti ve dosyaları indirme

Çalışma bitince oluşturulan ZIP dosyasını bu sohbete yükleyin. **FAST çıktılarını makalenin nihai deney bulguları olarak göndermeyin; PAPER modunda yeniden çalıştırın.**

In [ ]:
manifest = {
    'experiment': 'two_dataset_outer_cv_calibration_and_cost',
    'mode': MODE, 'outer_folds':OUTER_FOLDS,'inner_folds':INNER_FOLDS,
    'search_iter':SEARCH_ITER,'random_state':RANDOM_STATE,
    'cost_ratios': FN_COST_RATIOS,
    'datasets': {name:{'n':int(len(X)),'features':int(X.shape[1]),
                       'risk_prevalence':float(y.mean())} for name,(X,y,_) in datasets.items()},
    'versions':{'python':platform.python_version(),'numpy':np.__version__,
                'pandas':pd.__version__,'sklearn':sklearn.__version__,
                'xgboost':xgboost.__version__,'scipy':scipy.__version__},
    'notes': ['Calibrators use training folds only; threshold selection uses outer-training OOF predictions.',
              'Bootstrap compares fixed outer-fold predictions; it does not include training variability.',
              'Second dataset is small and historical; do not claim broad contemporary generalization.']
}
(OUTPUT_DIR/'manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
from zipfile import ZipFile,ZIP_DEFLATED
zip_path = OUTPUT_DIR.parent / ('kredi_ek_deney_sonuclari_'+MODE.lower()+'.zip')
with ZipFile(zip_path,'w',compression=ZIP_DEFLATED) as zout:
    for f in OUTPUT_DIR.iterdir():
        if f.is_file(): zout.write(f,arcname=f.name)
print('ZIP hazır:',zip_path)
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print('Google Colab dışında çalıştırılıyor; ZIP disk üzerinde bırakıldı.')

## Çalışma sonrasında makaleye uygulanacak revizyonlar

1. İki veri kümesindeki sonuçlar ve sınıf ağırlığı ablasyonu üzerinden genel performans sıralamasının **istikrarlı olup olmadığı** incelenecek; yalnızca eşit olan metrikleri anlatmakla yetinilmeyecek.
2. Bağımsız dış kat ve iç kalibrasyon mimarisi yöntem diyagramıyla gösterilecek; eski tek holdout sonuçları *pilot çalışma/önceki analiz* olarak ayrıştırılacak.
3. Kalibrasyon için yalnızca Brier değil **log-loss, ECE ve maliyet kararları** ayrı ayrı tartışılacak. ECE ve Brier'in farklı kavramlar olduğu belirtilecek.
4. SHAP–LIME yöntemleri için aynı tahmin ölçeği, kategorik değişken türleri ve örneklemler üzerinden ikinci bir **özel XAI deneyi** planlanacak. Bu notebook açıklama analizi üretmez; eski XAI sonuçları yeni model protokolüyle doğrulanmadan taşınmamalıdır.
5. Alman veri kümesi 1970'lerden geldiği ve hedefi kredi uyumsuzluğu olduğu için Tayvan kart temerrüdü ile doğrudan eşdeğer sonuç olarak sunulmayacak.
6. Makalenin başlığı ve özeti, yeni sonuçlar doğrulanmadan değiştirilip sayı eklenmeyecek.

**Kaynaklar:** UCI Default of Credit Card Clients (Yeh & Lien, 2009; repository ID 350), UCI South German Credit (repository ID 573; DOI: 10.24432/C5QG88).